# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [1]:
# Start by installing the dependencies
!pip install python-frontmatter getorg --upgrade
import frontmatter
import glob
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut

Iywidgets and ipyleaflet support disabled. You must be in a Jupyter notebook to use this feature.
Error raised:
No module named 'ipyleaflet'
Check that you have enabled ipyleaflet in Jupyter with:
    jupyter nbextension enable --py ipyleaflet


In [2]:
# Collect the Markdown files
g = glob.glob("_talks/*.md")

In [3]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="academicpages.github.io")
location_dict = {}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [4]:
# Perform geolocation
for file in g:
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Press on if the location is missing, empty, or virtual
    location = str(data.get('location') or '').strip()
    if not location or any(k in location.lower() for k in ('virtual', 'online', 'zoom')):
        continue

    # Prepare the description
    title = str(data.get('title') or '').strip()
    venue = str(data.get('venue') or '').strip()
    description = f"{title}<br />{venue}; {location}"

    # Geocode the location and report the status
    try:
        result = geocoder.geocode(location, timeout=TIMEOUT)
        if result is None:
            print(f"Error: no geocode result for input {location}")
            continue
        location_dict[description] = result
        print(description, result)
    except ValueError as ex:
        print(f"Error: geocode failed on input {location} with message {ex}")
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on input {location} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {location} with message {ex}")

Intelligent toolpath generation strategies for free-form machining operations<br />International Machining Conference; Istanbul, Turkiye İstanbul, Fatih, İstanbul, Marmara Bölgesi, 34122, Türkiye


Chico State Joins Community Push for Personal Protective Equipment<br />Chico State Today; Chico, CA Chico, Butte County, California, United States


Paper-based touch pads with a reduced number of multiplexed wires<br />ASME International Mechanical Engineering Congress and Exposition (IMECE); San Diego, CA, USA San Diego, San Diego County, California, United States


The Use of Mixed Reality for Manufacturing<br />National Institute of Standards and Technology - NIST; Gaithersburg, MD Gaithersburg, Montgomery County, Maryland, United States


In [5]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)

'Written map to talkmap/'